In [24]:
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import RBFInterpolator

def plot_3d_spline_regression(x1=None, x2=None, y=None, smoothing=1.0, n_points=30, filename="3d_spline.png"):
    """
    Fits a 3D thin-plate smoothing spline surface with roughness penalty and exports a PNG image.
    
    Parameters:
        smoothing: Penalty parameter for surface roughness/curvature (higher = smoother surface).
    """
    # Generate synthetic data with Y spread matching X (-5 to 5) if not provided
    if x1 is None or x2 is None or y is None:
        rng = np.random.default_rng(42)
        x1 = rng.uniform(-5, 5, 100)
        x2 = rng.uniform(-5, 5, 100)
        y = 4 * np.sin(np.sqrt(x1**2 + x2**2)) + rng.normal(scale=0.5, size=100)

    # Prepare 2D input matrix for SciPy RBFInterpolator
    X_train = np.column_stack((x1, x2))

    # Thin Plate Spline with curvature penalty controlled by `smoothing`
    spline = RBFInterpolator(
        X_train, 
        y, 
        kernel='thin_plate_spline', 
        smoothing=smoothing
    )

    # Create grid for surface prediction
    x1_range = np.linspace(x1.min(), x1.max(), n_points)
    x2_range = np.linspace(x2.min(), x2.max(), n_points)
    x1_grid, x2_grid = np.meshgrid(x1_range, x2_range)

    grid_points = np.column_stack((x1_grid.ravel(), x2_grid.ravel()))
    y_pred = spline(grid_points).reshape(x1_grid.shape)

    fig = go.Figure()

    # Scatter of original data points
    fig.add_trace(go.Scatter3d(
        x=x1, y=x2, z=y,
        mode='markers',
        marker=dict(size=4, color='red'),
        name='Data points'
    ))

    # Single-color surface with 3D shading
    fig.add_trace(go.Surface(
        x=x1_grid, y=x2_grid, z=y_pred,
        colorscale=[[0, 'royalblue'], [1, 'royalblue']],
        showscale=False,
        opacity=0.85,
        lighting=dict(
            ambient=0.4,
            diffuse=0.8,
            fresnel=0.2,
            specular=0.5,
            roughness=0.5
        ),
        lightposition=dict(x=100, y=200, z=150),
        name='Smoothing spline surface'
    ))

    # Camera configuration
    camera = dict(
        eye=dict(x=1.3, y=1.3, z=1.1),
    )

    fig.update_layout(
        title=dict(
            text='Spline regression',
            x=0.5,
            y=0.92,             # Lower title slightly closer to the scene
            xanchor='center',
            yanchor='top',
            font=dict(size=20)
        ),
        margin=dict(t=50, b=30, l=30, r=30), # Reduce top padding around canvas
        scene=dict(
            xaxis_title='X1',
            yaxis_title='X2',
            zaxis_title='Y',
            camera=camera
        ),
        width=800,
        height=700
    )

    # Save output as PNG
    fig.write_image(filename)
    fig.show()

    return fig, spline

In [25]:
fig, model = plot_3d_spline_regression(filename='./beamer/images/3d_spline.png')

In [40]:
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression

def plot_2d_classification_boundaries(x1=None, x2=None, y=None, n_points=200, filename="./beamer/images/classification_boundaries.png"):
    """
    Fits a linear multinomial classifier for 3 overlapping classes
    and plots decision boundary lines with distinct marker shapes in the X1-X2 plane.
    """

    # Generate synthetic data with fewer points (20 per class) and overlap
    if x1 is None or x2 is None or y is None:
        rng = np.random.default_rng(26)
        n_per_class = 10

        # Centered close together with high std dev for overlap
        x1_c0 = rng.normal(-0.8, 1.8, n_per_class)
        x2_c0 = rng.normal(-0.8, 1.8, n_per_class)

        x1_c1 = rng.normal(1.0, 1.8, n_per_class)
        x2_c1 = rng.normal(0.8, 1.8, n_per_class)

        x1_c2 = rng.normal(-0.2, 1.8, n_per_class)
        x2_c2 = rng.normal(1.8, 1.8, n_per_class)

        x1 = np.concatenate([x1_c0, x1_c1, x1_c2])
        x2 = np.concatenate([x2_c0, x2_c1, x2_c2])
        y = np.array([0] * n_per_class + [1] * n_per_class + [2] * n_per_class)

    X_train = np.column_stack((x1, x2))

    # Fit linear classifier
    clf = LogisticRegression(solver='lbfgs')
    clf.fit(X_train, y)

    # Create grid for decision boundary
    x1_range = np.linspace(x1.min() - 1, x1.max() + 1, n_points)
    x2_range = np.linspace(x2.min() - 1, x2.max() + 1, n_points)
    x1_grid, x2_grid = np.meshgrid(x1_range, x2_range)
    grid_points = np.column_stack((x1_grid.ravel(), x2_grid.ravel()))

    Z = clf.predict(grid_points).reshape(x1_grid.shape)

    fig = go.Figure()

    # Decision boundaries as lines only
    fig.add_trace(go.Contour(
        x=x1_range, y=x2_range, z=Z,
        showscale=False,
        contours=dict(
            coloring='none',
            showlines=True,
            start=0,
            end=2,
            size=1
        ),
        line=dict(width=2, color='black', dash='dash'),
        name='Decision boundary'
    ))

    # Scatter points with distinct shapes and larger marker size for clarity
    colors = ['red', 'green', 'blue']
    symbols = ['circle', 'square', 'triangle-up']
    class_names = ['Class 0', 'Class 1', 'Class 2']

    for cls in np.unique(y):
        mask = y == cls
        fig.add_trace(go.Scatter(
            x=x1[mask], y=x2[mask],
            mode='markers',
            marker=dict(
                size=10,
                symbol=symbols[cls],
                color=colors[cls],
                line=dict(width=1, color='black')
            ),
            name=class_names[cls]
        ))

    fig.update_layout(
        title=dict(
            text='Linear classification',
            x=0.5,             # Center title horizontally
            xanchor='center',
            font=dict(size=20) # Enlarged title font
        ),
        xaxis=dict(
            title='X1',
            title_font=dict(size=18),
            tickfont=dict(size=14)
        ),
        yaxis=dict(
            title='X2',
            title_font=dict(size=18),
            tickfont=dict(size=14)
        ),
        width=800,
        height=700,
        plot_bgcolor='white',
        legend=dict(
            font=dict(size=18)
        )
    )

    fig.write_image(filename)
    fig.show()

    return fig, clf

In [41]:
plot_2d_classification_boundaries()

(Figure({
     'data': [{'contours': {'coloring': 'none', 'end': 2, 'showlines': True, 'size': 1, 'start': 0},
               'line': {'color': 'black', 'dash': 'dash', 'width': 2},
               'name': 'Decision boundary',
               'showscale': False,
               'type': 'contour',
               'x': {'bdata': ('IY6igHHvHMAKzZcJF6wcwPQLjZK8aB' ... 'os5RZAScmukYYoF0BgirkI4WsXQA=='),
                     'dtype': 'f8'},
               'y': {'bdata': ('105/ZB/uFcCL7vku7asVwD6OdPm6aR' ... 'uDAh1ATSypALZEHUCYjC426IYdQA=='),
                     'dtype': 'f8'},
               'z': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'EBAQEBAQEBAQEBAQEBAQEBAQEBAQ=='),
                     'dtype': 'i1',
                     'shape': '200, 200'}},
              {'marker': {'color': 'red', 'line': {'color': 'black', 'width': 1}, 'size': 10, 'symbol': 'circle'},
               'mode': 'markers',
               'name': 'Class 0',
               'type': 'scatter',
               'x': {'bda